# 06 · Errors & Resource Management — **Depth**

> **Pull this notebook when:** you handle failures or manage resources — file/DB handles for Mara's
> memory, guarding generation against bad input, cleaning up no matter what.

Depth here is **control-flow under exceptions** and the context-manager protocol. Predict first.

---

## 6.1 — Predict: can `finally` change what's returned?

```python
def f():
    try:
        return "try"
    finally:
        return "finally"

def g():
    try:
        return "try"
    finally:
        print("  cleanup")   # runs, but no return here
```
What does `f()` return? What does `g()` return (and what prints)?

In [ ]:
def f():
    try:
        return "try"
    finally:
        return "finally"

def g():
    try:
        return "try"
    finally:
        print("  cleanup")

print(f())
print(g())

<details>
<summary>▶ Reveal</summary>

`f()` returns `"finally"`; `g()` prints `cleanup` then returns `"try"`.

`finally` **always runs** — even after a `return` in the `try`. In `f`, the `try` block sets up to
return `"try"`, but `finally` runs before the function actually exits, and its *own* `return` wins,
overriding the pending one. That's a genuine gotcha: a `return` in `finally` silently swallows whatever
`try` (or even an exception!) was about to do. In `g`, `finally` only does cleanup with no return, so
the `try`'s `"try"` stands.

The lesson: `finally` is for cleanup that must happen regardless (closing files, releasing locks).
**Never `return` from `finally`** — it masks results and exceptions. Know the behavior so you recognize
the bug.

</details>

## 6.2 — Predict: which `except` catches it?

Exceptions form a hierarchy; a parent catches its children. `except` clauses are tried **top to
bottom**, first match wins.

```python
def classify(e):
    try:
        raise e
    except ValueError:
        return "value"
    except Exception:
        return "general"

print(classify(ValueError("x")))        # ?
print(classify(KeyError("x")))          # ? KeyError is an Exception, not a ValueError
print(classify(TypeError("x")))         # ?
```

In [ ]:
def classify(e):
    try:
        raise e
    except ValueError:
        return "value"
    except Exception:
        return "general"

print(classify(ValueError("x")))
print(classify(KeyError("x")))
print(classify(TypeError("x")))

<details>
<summary>▶ Reveal</summary>

`value`, `general`, `general`.

`ValueError` matches the first clause. `KeyError` and `TypeError` aren't `ValueError`s, so they fall
through to `except Exception` (their common ancestor). Order matters: if you put `except Exception`
*first*, it would catch everything and the `ValueError` clause would be dead code (Python may even warn).
Always order from **most specific to most general**. And catching `Exception` (or bare `except:`) too
eagerly hides bugs — catch the narrowest type that matches what you're actually handling, the same
point from the fluency `hashables` exercise.

</details>

## 6.3 — Predict: a context manager that swallows errors

`with` calls `__enter__` on entry and `__exit__` on exit — *always*, even on exception. If `__exit__`
returns a **truthy** value, it **suppresses** the exception. Predict.

```python
class Suppress:
    def __enter__(self): return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f"  exit saw: {exc_type.__name__ if exc_type else None}")
        return True            # suppress whatever happened

print("before")
with Suppress():
    raise ValueError("boom")
    print("unreachable")
print("after")                 # does this run?
```

In [ ]:
class Suppress:
    def __enter__(self): return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f"  exit saw: {exc_type.__name__ if exc_type else None}")
        return True

print("before")
with Suppress():
    raise ValueError("boom")
    print("unreachable")
print("after")

<details>
<summary>▶ Reveal</summary>

```
before
  exit saw: ValueError
after
```

The `raise` triggers `__exit__`, which receives the exception's type/value/traceback. Returning `True`
tells Python "I handled it" — the exception is **suppressed**, and execution continues normally after
the `with` block, so `"after"` prints. If `__exit__` returned `False` (or `None`), the exception would
propagate out. This is the mechanism behind things like `contextlib.suppress`. The `exc_type` being
`None` when no error occurred is how `__exit__` distinguishes clean exit from error exit — which you'll
use to commit-vs-rollback in the milestone.

</details>

## 6.4 — Build: a context manager that times its block

Build `Timing` so that `with Timing(log) as t:` appends `"start"` to `log` on entry and `"end"` on
exit (even if the body raises — but it should NOT suppress the exception). Don't use `contextlib`;
implement `__enter__`/`__exit__` directly.

In [ ]:
class Timing:
    def __init__(self, log):
        self.log = log
    def __enter__(self):
        # YOUR CODE HERE — append "start", return self
        pass
    def __exit__(self, exc_type, exc_val, exc_tb):
        # YOUR CODE HERE — append "end"; do NOT suppress (return False/None)
        pass

# Tests
log = []
with Timing(log):
    log.append("work")
assert log == ["start", "work", "end"]

# __exit__ runs on error, and the error still propagates (not suppressed)
log2 = []
raised = False
try:
    with Timing(log2):
        raise ValueError("boom")
except ValueError:
    raised = True
assert raised
assert log2 == ["start", "end"]     # cleanup happened despite the error
print("6.4 passed")

<details>
<summary>▶ Why this guarantees cleanup</summary>

`__exit__` runs no matter how the block ends — normal completion or exception — which is the whole
point of `with`: guaranteed teardown. Returning `False`/`None` means "I didn't handle the exception,"
so it propagates after cleanup (contrast 6.3's `True`). This is exactly what `with open(...)` does to
guarantee the file closes even if your code throws mid-block. For Mara's memory you'll wrap DB
connections this way so they always close.

</details>

## 6.5 — Four ways to handle a possibly-missing key (and when each)

Counting word frequencies, a key may not exist yet. Implement the increment four ways and note the
trade-offs (answer). Each function takes a list of words and returns a count dict.

(implement all four below)

In [ ]:
from collections import defaultdict

def via_try(words):
    counts = {}
    for w in words:
        # YOUR CODE HERE — try counts[w] += 1, except KeyError set to 1
        pass
    return counts

def via_get(words):
    counts = {}
    for w in words:
        # YOUR CODE HERE — counts[w] = counts.get(w, 0) + 1
        pass
    return counts

def via_setdefault(words):
    counts = {}
    for w in words:
        # YOUR CODE HERE — counts.setdefault(w, 0); then += 1
        pass
    return counts

def via_defaultdict(words):
    counts = defaultdict(int)
    for w in words:
        # YOUR CODE HERE — counts[w] += 1
        pass
    return dict(counts)

# Tests
words = ["a", "b", "a", "c", "a"]
expected = {"a": 3, "b": 1, "c": 1}
assert via_try(words) == expected
assert via_get(words) == expected
assert via_setdefault(words) == expected
assert via_defaultdict(words) == expected
print("6.5 passed")

<details>
<summary>▶ When each</summary>

- **`try/except`** — works, but exceptions for *expected* control flow are slow and noisy. Reserve
  `try` for genuinely exceptional cases, not "the key might be new" (which is normal).
- **`.get(k, 0) + 1`** — clean, explicit, no import. The idiom you settled on in fluency. Great default
  for counting.
- **`setdefault`** — its real strength is *grouping* (`setdefault(k, []).append(...)`), where the
  default is a container you then mutate; for plain counting it's clunkier than `.get`.
- **`defaultdict(int)`** — cleanest for heavy counting/grouping; the missing-key default is automatic
  so the loop body is just `counts[w] += 1`. The cost is it's a slightly different type (and silently
  creates keys on read), so convert back with `dict(...)` when handing it out.

The progression `get` → `setdefault` → `defaultdict` → `Counter` is "the same need, increasingly
specialized tools." Knowing *why* each exists is the understanding; `defaultdict`/`Counter` mostly
supersede the manual versions once you're allowed the import.

</details>

## ★ Milestone 06 — A transaction context manager with rollback

Synthesize: a context manager that manages a resource with **commit on success, rollback on error**,
plus a custom exception hierarchy. Model a tiny in-memory "account" you can debit. Build:

- exceptions: `BankError(Exception)`, and `InsufficientFunds(BankError)` (a subclass)
- `Transaction(account)` — a context manager. On entry, snapshot the balance. On exit: if the block
  raised, **roll back** to the snapshot; if it succeeded, keep the changes (commit). It should NOT
  suppress the exception.
- `account` is just a dict like `{"balance": 100}`; the block debits it directly.

Tests: a successful transaction commits; one that raises `InsufficientFunds` rolls the balance back and
the error still propagates; catching `BankError` catches the `InsufficientFunds` subclass.

(build it below)

In [ ]:
class BankError(Exception):
    pass

class InsufficientFunds(BankError):
    pass

class Transaction:
    def __init__(self, account):
        self.account = account
    def __enter__(self):
        # YOUR CODE HERE — snapshot the current balance, return self
        pass
    def __exit__(self, exc_type, exc_val, exc_tb):
        # YOUR CODE HERE — if an exception occurred, restore the snapshot (rollback)
        # do NOT suppress (return False)
        pass

def debit(account, amount):
    # helper the block calls: raise InsufficientFunds if not enough, else subtract
    if amount > account["balance"]:
        raise InsufficientFunds(f"need {amount}, have {account['balance']}")
    account["balance"] -= amount

# Tests
# success commits
acct = {"balance": 100}
with Transaction(acct):
    debit(acct, 30)
    debit(acct, 20)
assert acct["balance"] == 50

# failure rolls back, and the error propagates
acct2 = {"balance": 100}
caught = False
try:
    with Transaction(acct2):
        debit(acct2, 40)        # ok, balance 60
        debit(acct2, 1000)      # raises — should roll back BOTH debits
except InsufficientFunds:
    caught = True
assert caught
assert acct2["balance"] == 100   # rolled all the way back to the snapshot

# catching the PARENT catches the subclass
acct3 = {"balance": 10}
caught_parent = False
try:
    with Transaction(acct3):
        debit(acct3, 999)
except BankError:               # parent of InsufficientFunds
    caught_parent = True
assert caught_parent
assert acct3["balance"] == 10
print("Milestone 06 passed")

<details>
<summary>▶ The pieces working together</summary>

`__enter__` snapshots state; `__exit__` checks `exc_type is not None` — the signal that the block
raised (from 6.3) — and if so restores the snapshot, giving all-or-nothing semantics: the second debit
failing undoes the first too, because rollback returns to the *entry* balance. Returning `False` lets
the exception propagate after cleanup (6.4). The exception hierarchy means callers can catch
`InsufficientFunds` specifically *or* `BankError` broadly (6.2). This commit/rollback-via-context-manager
pattern is exactly how you'd wrap writes to Mara's SQLite memory so a mid-write failure doesn't leave it
half-updated.

</details>